In [5]:
import sys
import os

# Agregar el directorio padre al sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pickle
import sqlite3

import json

import mlp.promok_functions as pk
import mlp.promok_models_config as pkconfig
from mlp.dataframe_tracker import *

import cashia_core.data_channel.promok_datapipe_func as pkdp
import cashia_core.data_channel.pm_datapipeline as pmdp


from datetime import datetime

In [6]:
import sys, pandas as pd, numpy as np, sklearn
print(sys.executable)
print("pandas", pd.__version__)
print("numpy", np.__version__)
print("sklearn", sklearn.__version__)

c:\Users\juan_\.conda\envs\cashia_env\python.exe
pandas 1.5.3
numpy 1.26.0
sklearn 1.2.2


In [7]:
date_and_time = datetime.now()
# Convierte la fecha y hora en una cadena con el formato deseado
str_date_and_time = date_and_time.strftime("%Y:%m:%d:%H:%M")

print(str_date_and_time)

all_stats = pk.ModelsStats()
all_stats.base_stats

2026:04:16:18:01


dict_values(['16/04/2026 18:01:37', 'Undefined', 'Undefined', 'Undefined', 'Undefined', 'Unknown', 'Unknown', -1, -1, -1, -1.0, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, None, -1, 'Unknown', 'Unknown', 'None', 'None', 'None'])

## 1 Lectura de datos de configuración

### 1.1 Datos categóricos

In [8]:
print('Configuration file: ', pkconfig.configuration_file_name)
all_features = pd.read_excel("./config/"+pkconfig.configuration_file_name, index_col=0)
features = all_features[all_features[pkconfig.conf_column] == 'Sí']

features_to_use = set(features['Característica'].values)
features_to_use

Configuration file:  CashIA_ConfFile.xlsx


c:\Users\juan_\.conda\envs\cashia_env\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


{'CP',
 'CP_Latitud',
 'CP_Longitud',
 'Dependientes',
 'Edad Solicitud',
 'Egresos',
 'Genero',
 'Incobrable',
 'Ingresos',
 'Mes inicial',
 'Monto',
 'Score Agt',
 'TipoZona',
 'Unidad'}

In [9]:
categorical_features = set(features[(features['Tipo de dato'] == 'Categórico') & (features['Tipo de variable'] == 'Predictor')]['Característica'].values)
categorical_features

{'CP', 'Genero', 'TipoZona', 'Unidad'}

### 1.2 Datos cuantitativos

In [10]:
quantitative_features = set(features[(features['Tipo de dato'] == 'Cuantitativo') & (features['Tipo de variable'] == 'Predictor')]['Característica'].values)
quantitative_features

{'CP_Latitud',
 'CP_Longitud',
 'Dependientes',
 'Edad Solicitud',
 'Egresos',
 'Ingresos',
 'Mes inicial',
 'Monto',
 'Score Agt'}

In [11]:
monetary_features = features[features['Monetario'] == 1]['Característica']

### 1.3 Registro de las características de los datos en las estadísticas

In [12]:
categorical_list = list(categorical_features)
categorical_list.sort()

quantitative_list = list(quantitative_features)
quantitative_list.sort()

all_stats.base_stats.fields["Model"] = pkconfig.conf_column
all_stats.base_stats.fields["Features"] = json.dumps({"Categorical":categorical_list,"Quantitative":quantitative_list})
all_stats.base_stats.fields["Notes"] = pkconfig.notes
all_stats.base_stats.fields["SourceFile"] = pkconfig.data_file_name

In [13]:
all_stats.base_stats

dict_values(['16/04/2026 18:01:37', 'Undefined', 'Undefined', 'Undefined', 'Undefined', 'Unknown', 'Unknown', -1, -1, -1, -1.0, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, None, -1, '{"Categorical": ["CP", "Genero", "TipoZona", "Unidad"], "Quantitative": ["CP_Latitud", "CP_Longitud", "Dependientes", "Edad Solicitud", "Egresos", "Ingresos", "Mes inicial", "Monto", "Score Agt"]}', 'Unknown', "zip_digits = 4  Usando: standardizationSin remplazar campos vacios por 'Unknown'", 'Data Set CashiaV3NV_AgtProcessed.csv', 'NV_Agt'])

## 2 Lectura de los datos de entrenamiento

In [14]:
print('Data file: ', pkconfig.data_file_name)
data = pd.read_csv("./data/"+pkconfig.data_file_name, parse_dates=['Fecha Inicial'], index_col=0, dayfirst=True)
# Convertir a entero el índice
data.index = data.index.astype(int)
# Ordenar el DataFrame por el índice de menor a mayor
data = data.sort_index()
# pkdp.print_sizes(data)
data_tracker = DataFrameEvolutionTracker(data)
data_tracker.to_dataframe()

Data file:  Data Set CashiaV3NV_AgtProcessed.csv


,Step,Rows,Columns,Column Names
0,Initialization,38862,15,"[Fecha Inicial, Monto, Edad Solicitud, Genero,..."


### 2.1 Borramos la columna de 'Score' si está presente ya que es una variable 'Target' usada en otro contexto.

In [15]:
if 'Score' in data.columns:
    data = data.drop(columns=['Score'])

In [16]:
data.head()

,Fecha Inicial,Monto,Edad Solicitud,Genero,Dependientes,CP,Ingresos,Egresos,Unidad,Incobrable,Score Agt,TipoZona,CP_Latitud,CP_Longitud,Mes inicial
index,,,,,,,,,,,,,,,
7956,2023-02-08,3000,38,Femenino,3,75100,14000,6000,Puebla Oriente,0,0.969498,Rural,19.070500,-97.747100,2
10606,2023-03-22,3500,46,Femenino,0,90790,8000,3000,Puebla Norte,1,0.729343,Urbano,19.166083,-98.208617,3
22031,2024-03-06,3000,54,Femenino,2,90640,6000,4000,Puebla Norte,0,0.971848,Urbano,19.346720,-98.135020,3
23891,2023-04-26,3000,46,Femenino,0,74129,12000,1000,Puebla Norte,0,0.998528,Urbano,19.253833,-98.429783,4
29931,2023-09-29,3000,54,Femenino,0,94480,4000,2000,Cordoba,1,0.798548,Unknown,18.953017,-96.947383,9


### 2.2 solo tomamos los datos hasta la fecha indicada

In [17]:
print(data.shape)
data = data[data['Fecha Inicial'] <= pkconfig.max_date]
pkdp.print_sizes(data)
data_tracker.register_step("Filtered by max date", data)
data_tracker.to_dataframe()

(38862, 15)
Rows: 38862
Columns: 15
Max index: 229640


,Step,Rows,Columns,Column Names
0,Initialization,38862,15,"[Fecha Inicial, Monto, Edad Solicitud, Genero,..."
1,Filtered by max date,38862,15,"[Fecha Inicial, Monto, Edad Solicitud, Genero,..."


### 2.3 Registramos datos para las estadísticas

In [18]:
all_stats.base_stats.fields["Size"] = len(data)

## 4 Simplificación del CP a n cifras

In [19]:
if pkconfig.zip_digits <= 4:
    print("Simplificando CP")
    if 'CP' in features_to_use:
        # Guardamos el conjunto de CP para almacenarlos posteriormente en 
        # el campo data_parameters de un modelo
        training_cps = data[['CP']]
        training_cps = training_cps.drop_duplicates().reset_index(drop=True)
        
        pkdp.transform_CP(data,'CP', pkconfig.zip_digits)
        
    if 'CP Aval' in features_to_use:
        # Guardamos el conjunto de CP Aval para almacenarlos posteriormente en 
        # el campo data_parameters de un modelo
        training_cps_aval = data[['CP Aval']]
        training_cps_aval = training_cps_aval.drop_duplicates().reset_index(drop=True)
        training_cps_aval = training_cps_aval.rename(columns={'CP Aval':'CP'})
        
        pkdp.transform_CP(data,'CP Aval', pkconfig.zip_digits)

Simplificando CP


In [20]:
data[monetary_features].head()

,Monto,Ingresos,Egresos
index,,,
7956,3000,14000,6000
10606,3500,8000,3000
22031,3000,6000,4000
23891,3000,12000,1000
29931,3000,4000,2000


In [ ]:
converter = pmdp.CurrentValueConverter(pkconfig.get_mlp_resource_key("inflation_table"))

FileNotFoundError: [Errno 2] No such file or directory: 'G:\\My Drive\\19_Projects\\cashia\\data\\tablaDeInflacion.csv'

In [ ]:
for feature in monetary_features:
    data = converter.to_current_value(data, 'Fecha Inicial', feature)

In [ ]:
data[monetary_features].head()

In [ ]:
#data.to_excel("ProcessedData2.xlsx")

In [ ]:
converter_500 = pmdp.MultipleAdjuster(500)
data = converter_500.adjust_amount(data, 'Monto')

In [ ]:
data[monetary_features].head()

In [ ]:
# Monto promedio de los "buenos pagadores"
good_payer = data[data['Incobrable']==0]
good_payer['Monto'].mean()

In [ ]:
# Monto promedio de los "malos pagadores"
bad_payer = good_payer = data[data['Incobrable']==1]
bad_payer['Monto'].mean()

In [ ]:
# Promedio de "Incobrabilidad"
data['Incobrable'].mean()

In [ ]:
# Promedio del monto de préstamo
data['Monto'].mean()

## 3 Feature engineering directly with sklearn

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder

predictors = data.drop(columns=['Incobrable', 'Fecha Inicial'])

target = data['Incobrable']

preprocessor = make_column_transformer(
    (
        OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=99999999),
        list(categorical_features),
    ),
    (StandardScaler(), list(quantitative_features)),
)


## 6 Run models

In [ ]:
train_type = pk.BY_SIZE

In [ ]:
if train_type == pk.BY_SIZE:

    
    totals = data["Incobrable"].value_counts()
    
    train_init_date = data.iloc[0]['Fecha Inicial']
    train_end_date = data.iloc[-1]['Fecha Inicial']
    
    data = data.drop('Fecha Inicial', axis=1)
    
    all_stats.base_stats.fields['Paid'] = int(totals[0])
    all_stats.base_stats.fields['Unpaid'] = int(totals[1])
    all_stats.base_stats.fields['AccMin'] = float(max(totals[0],totals[1])/(totals[0]+totals[1])*100)
    
    all_stats.base_stats.fields["TestSize"] = pkconfig.test_size
    models_tester = pk.ModelsTester(data, pkconfig.models_to_test, 
                                    pkconfig.test_size, split=pk.BY_SIZE)
    
elif train_type == pk.BY_INDEX:
    dates = data[['Fecha Inicial']]
    data = data.drop('Fecha Inicial', axis=1)
    
    data_size = data.shape[0]
    train_size = int(data_size/2)
    test_init = int(data_size*4/5)
    test_size = int(data_size/5)

    print("Data size: ", data_size)
    print("Train size: ", train_size)
    print("Test init: ", test_init)
    print("Test size: ", test_size)

    test_data = data.iloc[test_init:test_init+test_size]

    train_init_date = dates.loc[data.index[0],'Fecha Inicial']
    train_end_date = dates.loc[data.index[train_size],'Fecha Inicial']
    test_init_date = dates.loc[data.index[test_init],'Fecha Inicial']
    test_end_date = dates.loc[data.index[test_init+test_size-1],'Fecha Inicial']

    print("Evalueating with:")
    print(f"Train from: {train_init_date} to {train_end_date}")
    print(f"Test from: {test_init_date} to {test_end_date}")

    all_stats.base_stats.fields["TrainInitDate"] = str(train_init_date)
    all_stats.base_stats.fields["TrainEndDate"] = str(train_end_date)
    all_stats.base_stats.fields["TestInitDate"] = str(test_init_date)
    all_stats.base_stats.fields["TestEndDate" ] = str(test_end_date)

    all_stats.base_stats.fields["Size"] = len(test_data)

    totals = test_data["Incobrable"].value_counts()
    all_stats.base_stats.fields['Paid'] = int(totals[0])
    all_stats.base_stats.fields['Unpaid'] = int(totals[1])

    all_stats.base_stats.fields['AccMin'] = float(max(totals[0],totals[1])/(totals[0]+totals[1])*100)
    
    models_tester = pk.ModelsTester(data, pkconfig.models_to_test, pkconfig.test_size,
                                    train_init=0, train_size=train_size, test_init=test_init, 
                                    test_size=test_size, split=pk.BY_INDEX)
    all_stats.base_stats.fields["TestSize"] =  models_tester.x_test.shape[0]/models_tester.data.shape[0]
else:
    train_init_date = data.iloc[0]['Fecha Inicial']
    train_end_date = pd.to_datetime('2022-12-22')

    test_init_date = train_end_date + pd.Timedelta(days=1)
    test_end_date = pd.to_datetime('2023-03-22')
    
    print("Evalueating with:")
    print(f"Train from: {train_init_date} to {train_end_date}")
    print(f"Test from: {test_init_date} to {test_end_date}")
    
  
    all_stats.base_stats.fields["TrainInitDate"] = str(train_init_date)
    all_stats.base_stats.fields["TrainEndDate"] = str(train_end_date)
    all_stats.base_stats.fields["TestInitDate"] = str(test_init_date)
    all_stats.base_stats.fields["TestEndDate" ] = str(test_end_date)

    
    models_tester = pk.ModelsTester(data, pkconfig.models_to_test,  
                                    train_end_date=train_end_date,
                                    test_end_date=test_end_date,
                                    split=pk.BY_DATE)
    
    all_stats.base_stats.fields["TestSize"] = models_tester.x_test.shape[0]/models_tester.data.shape[0]
    all_stats.base_stats.fields["Size"] = models_tester.data.shape[0]

    totals = models_tester.y_test.value_counts()
    all_stats.base_stats.fields['Paid'] = int(totals[0])
    all_stats.base_stats.fields['Unpaid'] = int(totals[1])

    all_stats.base_stats.fields['AccMin'] = float(max(totals[0],totals[1])/(totals[0]+totals[1])*100)

In [ ]:
data.shape

In [ ]:
models_tester.x_train.columns

In [ ]:
all_columns = list(data.columns)
all_columns.sort()
all_columns

### Correr los diferentes modelos (hacer el 'fit' con cada uno de ellos) con los datos de entrenamiento

In [ ]:
models_tester.run_all_models(preprocessor)

### Correr los modelos con los datos de prueba y desplegar las estadísticas

In [ ]:
for model in models_tester.models:
    print(f"********************* Results for model: { model.name} : {model.parameters} *********************")
    print("********************** TEST **********************")
    y_pred_test = model.pipeline.predict(models_tester.x_test)
    all_stats.add_and_print_stats(models_tester.y_test, y_pred_test, model.name, model.parameters, "Test")

    print("********************** TRAIN **********************")
    y_pred_train = model.pipeline.predict(models_tester.x_train)
    all_stats.add_and_print_stats(models_tester.y_train, y_pred_train, model.name, model.parameters, "Train")
    
    #if (model.name != "Random Forest") and (model.name != "Ada Boost"):
    if (model.name != "Ada Boost"):
        ### Estadisticas con el mejor umbral para el accurracy y monetary gain

        predictions = model.pipeline.predict_proba(models_tester.x_test)[:,1]
        acceptance_threshold, accurracy_list, best_threshold, best_accurracy, gl, bgt, bg =  \
                                                                        pk.find_best_threshold(predictions, models_tester.y_test)
        model.best_gain_threshold = bgt
        model.best_accurracy_threshold = best_threshold
        
        print("**************** Unbral para best accurracy *******************")
        print("Best threshold = ", best_threshold)
        pk.plot_threshold_evolution(acceptance_threshold, accurracy_list, "Accurracy of the model by changing the threshold",
                                    "Threshold","Accuracy")
        y_pred_accuracy = np.where(predictions > best_threshold, 1, 0)
        all_stats.add_and_print_stats(models_tester.y_test, y_pred_accuracy, model.name, model.parameters, 
                                      "Best accurracy threshold Test", True)

        ### Estadísticas de con la mejor ganancia monetária
        print("**************** Unbral para best monetary gain Test *******************")
        print("Best gain threshold = ", bgt)
        pk.plot_threshold_evolution(acceptance_threshold, gl, 
                                    "Monetary gain of the model by changing the threshold (Test)", 
                                    "Threshold", "Monetary gain")
        y_pred_monetary_test = np.where(predictions > bgt, 1, 0)
        print(f"Model configuration:{pkconfig.conf_column}")
        all_stats.add_and_print_stats(models_tester.y_test, y_pred_monetary_test, model.name, model.parameters, 
                                      "Best monetary gain threshold Test", True)
    
        ### Estadísticas con la mejor ganancia monetária datos de entrenamiento
        print("**************** Unbral para best monetary gain Train *******************")
        predictions = model.pipeline.predict_proba(models_tester.x_train)[:,1]
        acceptance_threshold, accurracy_list, best_threshold, best_accurracy, gl, bgt, bg = pk.find_best_threshold(predictions, models_tester.y_train)

        print("Best gain threshold = ", bgt, "with a monetary gain of", bg)
        pk.plot_threshold_evolution(acceptance_threshold, gl, "Monetary gain of the model by changing the threshold (Train)",
                                    "Threshold", "Monetary gain")

        y_pred_monetary_train = np.where(predictions > bgt, 1, 0)
        all_stats.add_and_print_stats(models_tester.y_train, y_pred_monetary_train, model.name, model.parameters,"Best monetary gain threshold Train")
    #model.min_max_stats = min_max_stats
    model.columns = models_tester.x_train.columns
    model.feature_engineering = pkconfig.feature_engineering
    model.features = all_features
    model.conf_column = pkconfig.conf_column
    model.data_parameters['zip_digits'] = pkconfig.zip_digits
    model.data_parameters['Training init date'] = train_init_date
    model.data_parameters['Trainig en date'] = train_end_date
    model.data_parameters['SourceFile'] = pkconfig.source_file_name
    model.data_parameters['Date'] = str_date_and_time
    if 'CP' in features_to_use:
        model.data_parameters['CP'] = training_cps
    if 'CP Aval' in features_to_use:
        model.data_parameters['CP Aval'] = training_cps_aval
    model.save_with_pickle()

In [ ]:
for model in models_tester.models:
    print(model)
    print("=====================================")

# Write statistics

## Write to data base

In [ ]:
con = sqlite3.connect("promocash_stats.db")
cur = con.cursor()
cur.executemany("INSERT INTO stats VALUES(?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", all_stats.values)
con.commit()

### Leer de la base de datos para escribir las estadísticas en la base de datos

In [ ]:
df = pd.read_sql_query("SELECT * FROM stats", con)
df

In [ ]:
time_now = datetime.now()

format_date_hour = "%Y-%m-%d %H:%M:%S"
date_and_hour = time_now.strftime(format_date_hour)

# Imprimir la fecha y hora
print("Ended at:", date_and_hour)

In [ ]:
con.commit()
con.close()

In [ ]:
df.index.name = 'index'
df.to_excel("./statistics/"+pkconfig.stats_file_name)

In [ ]:
# Escribir el archivo de data tracking
data_tracker.to_dataframe().to_csv("./data/" + pkconfig.conf_column + "_data_tracking_train.csv")

In [ ]:
test_result = models_tester.x_test.copy()
test_result["Prediction"] = y_pred_monetary_test
test_result["Incobrable"] = models_tester.y_test
test_result

In [ ]:
test_result.to_csv("./data/TestResult"+pkconfig.conf_column+".csv")

In [ ]:
preprocessor.fit(models_tester.x_train)
x_test_processed = preprocessor.transform(models_tester.x_test)

# Crear DataFrame a partir de los datos transformados
x_test_transformed_df = pd.DataFrame(
    data=x_test_processed,
    columns=models_tester.x_test.columns
)

In [ ]:
models_tester.x_test.columns

In [ ]:
x_test_processed[0]

In [ ]:
x_test_transformed_df["Prediccion"] = models_tester.y_test.to_numpy()

In [ ]:
models_tester.y_test

In [ ]:
x_test_transformed_df

In [ ]:
time_now = datetime.now()

date_and_hour = time_now.strftime(format_date_hour)

# Imprimir la fecha y hora
print("Ended at:", date_and_hour)

In [ ]:
# Hacer un sonido
if sys.platform.startswith("win"):
    import winsound
    winsound.Beep(1000, 500)  # Frecuencia de 1000 Hz, duración de 500 ms
    winsound.Beep(500, 500) 
    winsound.Beep(1000, 500) 
else:
    print('\a')  # Hace un beep en algunos sistemas UNIX
    
print('Configuration : ', pkconfig.conf_column)

In [ ]:
len(x_test_transformed_df)

In [ ]:
models_tester.y_test

In [ ]:

max_score_value = data['Score Agt'].max()
min_score_value = data['Score Agt'].min()
print("Min: ", min_score_value)
print("Max: ", max_score_value)

In [ ]:
n_groups = 100
bins = np.linspace(min_score_value, max_score_value, n_groups+1)
labels = list(range(1,n_groups+1))

In [ ]:
bins

In [ ]:
# x_test_transformed_df['Group'] = pd.cut(models_tester.x_test['Score Agt'], bins, labels=labels)

groups = pd.cut(
    models_tester.x_test['Score Agt'].to_numpy(),  # sin índice
    bins=bins,
    labels=labels
)

x_test_transformed_df['Group'] = groups  # groups ya no tiene índice de pandas


In [ ]:
intervals = [f"[{bins[i]:.3f}, {bins[i+1]:.3f}]" for i in range(0, n_groups)]

In [ ]:
x_test_transformed_df

In [ ]:
grouped_data_mean = x_test_transformed_df.groupby('Group').mean()
grouped_data_mean = grouped_data_mean.reset_index()

In [ ]:
grouped_data_mean = grouped_data_mean.rename(columns={"Prediccion":'% Predicción de incobrables'})

In [ ]:
grouped_data_mean.columns

In [ ]:
grouped_data_mean.plot.bar(x='Group', y='% Predicción de incobrables',  ylabel="% Predicción de incobrables",)

In [ ]:
grouped_data_mean[['Group', '% Predicción de incobrables']]

In [ ]:
grouped_data_count = x_test_transformed_df.groupby('Group').count()
grouped_data_count = grouped_data_count.reset_index()
grouped_data_count = grouped_data_count.rename(columns={'Prediccion':'No. de Créditos'})

In [ ]:
grouped_data_count

In [ ]:
grouped_data_count[['Group','No. de Créditos']]

In [ ]:
grouped_data_count.plot.bar(x='Group', y='No. de Créditos',  ylabel="No créditos",)

In [ ]:
grouped_data_sum = x_test_transformed_df.groupby('Group').sum()
grouped_data_sum = grouped_data_sum.reset_index()
grouped_data_sum = grouped_data_sum.rename(columns={'Prediccion':'No. Créditos rechazados'})

In [ ]:
grouped_data_sum[['Group','No. Créditos rechazados']]

In [ ]:
grouped_data_sum['No. Créditos'] = grouped_data_count['No. de Créditos']

In [ ]:
grouped_data_sum[['No. Créditos', 'No. Créditos rechazados']]

In [ ]:
grouped_data_sum['No Créditos aprobados'] = grouped_data_sum['No. Créditos'] - grouped_data_sum['No. Créditos rechazados']

In [ ]:
grouped_data_sum[['Group','No. Créditos', 'No. Créditos rechazados', 'No Créditos aprobados']]

In [ ]:
grouped_data_sum[['Group', 'No. Créditos rechazados', 'No Créditos aprobados']].plot.bar(x='Group', stacked=True, rot=0);

In [ ]:
zoom_data = grouped_data_sum[(grouped_data_sum['Group']>=40) & (grouped_data_sum['Group']<=60)]

In [ ]:
zoom_data[['Group','No. Créditos', 'No. Créditos rechazados', 'No Créditos aprobados']]

In [ ]:
zoom_data[['Group', 'No. Créditos rechazados', 'No Créditos aprobados']].plot.bar(x='Group', stacked=True, rot=0)

In [ ]:
#training_cps.to_excel("training_cps.xlsx")

In [ ]:
data.head()

In [ ]:
list(data.columns).sort()

In [ ]:
data.shape[0]